In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
import importlib.util, sys
import matplotlib.pyplot as plt

# ===================== 相对路径配置（核心修正） =====================
# 1. 确定当前Notebook所在目录（Python_for_Check）
CURRENT_DIR = Path.cwd()  # 等价于：d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\1_Code\Python_for_Check
# 2. 相对路径找runner文件：从Python_for_Check → 回到1_Code → 进入Python_for_Generate/Py → 找到runner文件
runner_path = CURRENT_DIR.parent / "Python_for_Generate" / "Py" / "Generate_Data_v2.4_runner.py"

# 3. 项目根目录（1_Code的上一级）：用于保存结果/图表
BASE_DIR = CURRENT_DIR.parent.parent  # 等价于：d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design

# 动态导入外部脚本（原逻辑保留）
spec = importlib.util.spec_from_file_location("gdrv2_runner", str(runner_path))
gdrv2 = importlib.util.module_from_spec(spec)
sys.modules["gdrv2_runner"] = gdrv2
spec.loader.exec_module(gdrv2)
generate_dataset_s2 = gdrv2.generate_dataset_s2
normalize_PTW_to_unit = gdrv2.normalize_PTW_to_unit

# 结果/图表保存路径（基于项目根目录，自动创建）
OUT_DIR = BASE_DIR / "2_Data" / "Generate_Data"
FIG_DIR = BASE_DIR / "3_Figures" / "Generate_Data_v2.4_recovery"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 验证路径是否正确（打印检查，新手友好）
print(f"✅ 当前Notebook目录：{CURRENT_DIR}")
print(f"✅ Runner文件路径：{runner_path}，文件是否存在：{runner_path.exists()}")
print(f"✅ 结果保存路径：{OUT_DIR}")
print(f"✅ 图表保存路径：{FIG_DIR}")

# 若打印"文件是否存在：False"，请检查：
# 1. Python_for_Generate/Py 目录是否存在
# 2. Generate_Data_v2.4_runner.py 文件名是否完全一致（大小写/后缀）

✅ 当前Notebook目录：d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\1_Code\Python_for_Check
✅ Runner文件路径：d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\1_Code\Python_for_Generate\Py\Generate_Data_v2.4_runner.py，文件是否存在：True
✅ 结果保存路径：d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Generate_Data
✅ 图表保存路径：d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\Generate_Data_v2.4_recovery


In [4]:
def build_X(df):
    # 对每一行数据归一化P/T/W
    Pn, Tn, Wn = zip(*[normalize_PTW_to_unit(r.P, r.T, r.W) for r in df.itertuples()])
    # 将label转为数值：self=1.0，stranger=0.0
    label = (df['label'] == 'self').astype(float).values
    # 拼接成4维特征矩阵
    X = np.column_stack([Pn, Tn, Wn, label])
    return X

In [5]:
def fit_and_score_gp(X_train, y_train, X_test, y_test):
    # 定义GP核函数（和参数生成时一致）
    kernel = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-5)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True)
    # 训练GP模型
    gp.fit(X_train, y_train)
    # 预测测试集（返回预测值和标准差）
    y_pred, std = gp.predict(X_test, return_std=True)
    # 计算RMSE（误差，越小越好）
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # 计算皮尔逊相关系数r（相关性，越接近1越好）
    try:
        r, _ = pearsonr(y_test, y_pred)
    except Exception:
        r = np.nan  # 处理极端情况（比如所有值相同）
    return rmse, r

In [6]:
def run_recovery(sizes=(20,50,100), v_noises=(0.2,0.5,1.0,2.0), repeats=5):
    rows = []  # 存储所有实验结果
    # 遍历所有实验条件：被试数量 + 噪声水平 + 重复次数
    for n_subj in sizes:
        for v_noise in v_noises:
            for rep in range(repeats):
                # 生成模拟数据集
                df, gen = generate_dataset_s2(
                    n_subjects=n_subj, 
                    trials_per_sub=60, 
                    w_gp=0.5, 
                    v_noise=v_noise, 
                    a_noise=0.5, 
                    seed=1000+rep  # 固定种子，结果可复现
                )
                # 构建特征矩阵X
                X = build_X(df)
                # 提取真实的v和a作为目标值
                y_v = df['v'].values
                y_a = df['a'].values
                # 划分训练/测试集（80%训练，20%测试）
                X_train, X_test, yv_train, yv_test = train_test_split(X, y_v, test_size=0.2, random_state=42)
                _, _, ya_train, ya_test = train_test_split(X, y_a, test_size=0.2, random_state=42)
                # 训练GP并评估v的恢复效果
                rmse_v, r_v = fit_and_score_gp(X_train, yv_train, X_test, yv_test)
                # 训练GP并评估a的恢复效果
                rmse_a, r_a = fit_and_score_gp(X_train, ya_train, X_test, ya_test)
                # 保存本次实验结果
                rows.append({
                    'n_subjects': n_subj, 'v_noise': v_noise, 'rep': rep,
                    'rmse_v': rmse_v, 'r_v': r_v, 'rmse_a': rmse_a, 'r_a': r_a
                })
                # 打印实时进度（新手友好）
                print(f"完成：被试数={n_subj} | 噪声={v_noise} | 重复={rep} → "
                      f"v误差={rmse_v:.3f} | v相关={r_v:.3f} | a误差={rmse_a:.3f} | a相关={r_a:.3f}")
    
    # 保存结果到CSV
    res = pd.DataFrame(rows)
    res_path = OUT_DIR / 'recovery_results_v2.4.csv'
    res.to_csv(res_path, index=False)
    print(f"\n所有结果已保存到：{res_path}")

    # ===================== Notebook 适配：可视化优化 =====================
    %matplotlib inline  # 让图表内嵌显示（Notebook 核心）
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']  # 适配中文可换 'SimHei'
    plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
    plt.rcParams['figure.dpi'] = 100  # 图表清晰度

    # 绘制4个评估指标的折线图
    for metric in ['rmse_v','r_v','rmse_a','r_a']:
        plt.figure(figsize=(10, 6))  # 放大图表，更清晰
        for n_subj in sizes:
            subset = res[res['n_subjects']==n_subj]
            # 计算每个噪声水平的均值（多次重复的平均）
            means = subset.groupby('v_noise')[metric].mean()
            # 绘制折线图（加网格，新手易读）
            plt.plot(means.index, means.values, marker='o', linewidth=2, label=f'被试数={n_subj}')
        # 图表美化（新手友好）
        plt.xlabel('v参数噪声水平', fontsize=12)
        plt.ylabel(metric, fontsize=12)
        plt.title(f'{metric} 随噪声水平和被试数的变化', fontsize=14)
        plt.legend(fontsize=10)
        plt.grid(alpha=0.3)  # 加网格
        # 保存图表（避免截断）
        plt.savefig(FIG_DIR / f'{metric}_by_vnoise.png', dpi=200, bbox_inches='tight')
        plt.show()  # Notebook 中显示图表
    
    print(f"\n所有图表已保存到：{FIG_DIR}")
    return res  # 返回结果DataFrame，方便后续分析

In [ ]:
# 运行参数恢复实验（可修改参数，比如减少repeats加快速度）
# 新手建议先试小参数：sizes=(20,50), v_noises=(0.2,0.5), repeats=2
recovery_results = run_recovery(
    sizes=(20,50,100),  # 被试数量
    v_noises=(0.2,0.5,1.0,2.0),  # v参数噪声水平
    repeats=5  # 重复次数（越多结果越稳，也越慢）
)

# 可选：查看结果前5行（新手验证是否运行成功）
print("\n实验结果预览：")
print(recovery_results.head())

c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warni

完成：被试数=20 | 噪声=0.2 | 重复=0 → v误差=1.212 | v相关=0.908 | a误差=0.486 | a相关=0.704


c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warning

完成：被试数=20 | 噪声=0.2 | 重复=1 → v误差=0.201 | v相关=0.987 | a误差=0.522 | a相关=0.706


c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warning

...

c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warning

完成：被试数=50 | 噪声=2.0 | 重复=3 → v误差=2.020 | v相关=0.506 | a误差=0.512 | a相关=0.637


c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warning

完成：被试数=50 | 噪声=2.0 | 重复=4 → v误差=2.095 | v相关=0.457 | a误差=0.483 | a相关=0.660


c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warning